In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from primaite.network.generator import NetworkGenerator
from primaite.agents.llm.observation import ObservedState
from pathlib import Path
import os
from dotenv import load_dotenv

/home/sam/projects/PrimAITE/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sam/projects/PrimAITE/.venv/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
2024-08-30 16:26:22.293587: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-30 16:26:22.327948: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler f

In [3]:
# constants
NUMBER_OF_NODES = 30
RANDOM_SEED = 165116
TRAINING_CONFIG_PATH = Path("../../../../agents") / "training_configs" / "do_nothing.yaml"
LAYDOWN_SAVE_PATH = Path("../../../../data") / "laydown_configs" / "hey_buddy.yaml"
SERVICE_NAMES = ["HTTP", "SSH", "FTP"]
PORTS_LIST = ["80", "22", "21"]

In [4]:
generator = NetworkGenerator(training_config_path=TRAINING_CONFIG_PATH, random_seed=1729, services=SERVICE_NAMES, ports=PORTS_LIST)
network = generator.network
generator.generate_iers()
generator.generate_green_pols()
generator.generate_red_pols()
generator.generate_laydown(LAYDOWN_SAVE_PATH)

target nide id,  23
target nide id,  24
target nide id,  19
target nide id,  9
target nide id,  28


In [5]:
observed_state = ObservedState(network=network)

In [6]:
from pydantic import BaseModel

load_dotenv() 
open_ai_key = os.getenv('OPEN_AI_KEY')

from primaite.agents.aegis.modules.openai import OpenAIClient
class Test(BaseModel):
    name: str
prompt = 'Extract my name from the following text: my name is dumdum'


openai = OpenAIClient(api_key=open_ai_key)


In [7]:
from primaite.agents.llm_openai.prompting import ACTION_INFO



In [8]:
from pydantic import BaseModel

from primaite.agents.llm_openai.prompting import AgentReasoningNodeSelection


REASON_ACTION_SPACE_NODE_SELECT = f"""
This is the initial configuration of the network:
{generator.network_description}

As an agent, you are able to influence the state of this node by switching it on or off, resetting it, patching software or patching any of its services.

Please think about the network configuration and the state of each node. Think about which nodes are most vulnerable to attack and which node requires action the most in order to stop the attack and prevent further spread. Provide your reasoning statement and select a node by name to perform a defensive action on. If no action is required because all is well, you can simply say 'NONE', but always provide a reasoning statement.

If action is not required at the moment, set NONE as the node_name, but always reason over the state of the network.
Note that actions are expensive and can negatively impact the environment if used improperly. For instance, a server which is turned off cannot receive requests from the users and will decrease the reward.  

If choosing to take an action, you must select a node by name from the following list: {list(generator.nodes_dict.values())}.


For your information, the following actions are available for selection later. Always take note of any action constraints outlined in the description provided.

{ACTION_INFO.format(service_names=generator.services)}

Your output should be in the following format:
{{'reasoning': 'Reason for node selection', 'node_name': 'NODE_NAME'}}
Reasoning and node selection: 
"""

openai.generate_model(prompt=REASON_ACTION_SPACE_NODE_SELECT, grammar=AgentReasoningNodeSelection)

AgentReasoningNodeSelection(reasoning='Node_7 is the most vulnerable node to attack as it has multiple connections with other nodes such as Node_11, Node_14, and Node_19. Taking defensive action on Node_7 by patching its software can help prevent the spread of any potential attacks.', node_name='Node_7')

In [10]:
from primaite.agents.llm_openai.prompting import AgentNodeAction


openai.generate_model(prompt=REASON_ACTION_SPACE_NODE_SELECT, grammar=AgentNodeAction)

AgentNodeAction(node_name='Node_10', node_property='SOFTWARE', property_action='PATCH', service_name='NONE')

primaite.agents.llm_openai.prompting.AgentNodeAction